# Code Repair Assistant -- Fine-Tuning and Benchmark (Parts D & E)

Fine-tunes **Qwen2.5-Coder-1.5B-Instruct** on the real MBPP-derived
code-repair dataset (built and sandbox-verified in Parts A/B), compares
**LoRA vs QLoRA vs DoRA**, runs **DPO** on top of the best SFT adapter, then
benchmarks every variant on the **held-out HumanEval broken-code set** with
pass@1 / pass@3 measured by actually executing each generated fix in the
Part A sandbox.

**Rules for this notebook**
- Run cells top to bottom. Sections free GPU memory before the next starts.
- Every printed number is measured at run time. Outputs ship empty; nothing
  is estimated or pre-filled.
- Each adapter is backed up to Drive immediately after its training finishes.
- Expected runtime: L4 GPU + High-RAM. Rough wall-clock at 1.5B: 6-12 min
  per SFT run at the default settings, 5-10 min for DPO, 4-8 min per
  evaluation arm. Reduce `N_EVAL` or `NUM_EPOCHS` for an even faster pass.

In [1]:
# ---- Part D, step 1: environment gate (run before anything else) ----
import os, subprocess

try:
    import torch
except ImportError:
    raise SystemExit("PyTorch is not available -- this is not a Colab GPU "
                     "runtime. Runtime > Change runtime type > GPU (L4).")

if not torch.cuda.is_available():
    raise SystemExit(
        "NO GPU DETECTED.\n"
        "This notebook requires a GPU runtime (expected: NVIDIA L4).\n"
        "Fix: Runtime > Change runtime type > Hardware accelerator: GPU, "
        "GPU type: L4, then restart and re-run this cell.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
with open("/proc/meminfo") as f:
    ram_gb = int(f.readline().split()[1]) / 1024**2

print(f"GPU        : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"System RAM : {ram_gb:.1f} GB")
print(subprocess.run(["nvidia-smi", "--query-gpu=driver_version,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

ALLOW_NON_L4 = False  # set True only if you accept a different GPU

if vram_gb < 20:
    raise SystemExit(
        f"GPU '{gpu_name}' has only {vram_gb:.1f} GB VRAM -- the LoRA (bf16) "
        "run needs ~20+ GB. Select an L4 (24 GB) runtime.")
if "L4" not in gpu_name and not ALLOW_NON_L4:
    raise SystemExit(
        f"Expected an NVIDIA L4, got '{gpu_name}'. If you intend to use it "
        "anyway, set ALLOW_NON_L4 = True above and re-run.")
if ram_gb < 30:
    raise SystemExit(
        f"Only {ram_gb:.1f} GB system RAM detected -- enable High-RAM: "
        "Runtime > Change runtime type > High-RAM, then re-run.")
print("\nEnvironment OK: L4-class GPU + High-RAM confirmed.")

GPU        : NVIDIA L4
VRAM       : 22.0 GB
System RAM : 53.0 GB
580.82.07, 23034 MiB

Environment OK: L4-class GPU + High-RAM confirmed.


In [ ]:
# ---- Part D, step 2a: dependencies (torch-safe install) ----
# Colab ships a matched torch+torchvision+CUDA build already. Installing
# transformers/peft/trl/bitsandbytes with no torch pin lets pip's resolver
# silently downgrade torch to satisfy an older dependency -- that mismatch
# is exactly what produces "cannot import name 'skip_code' from
# torch._C._dynamo.eval_frame". Fix: capture Colab's torch version BEFORE
# installing anything, then pin torch to that exact version in the same
# install command so pip is not allowed to touch it.
import torch as _torch
_TORCH_PIN = _torch.__version__.split("+")[0]
print(f"Colab's preinstalled torch: {_torch.__version__} "
      f"-- pinning torch=={_TORCH_PIN} so this install cannot change it")

%pip -q install "transformers==4.46.2" "peft==0.13.2" "trl==0.12.1" \
    "bitsandbytes==0.44.1" "accelerate==1.1.1" "datasets==3.1.0" \
    "triton==3.1.0" pandas "torch=={_TORCH_PIN}"

print("\nInstall finished. IMPORTANT: Runtime > Restart session now "
      "(not 'Disconnect and delete runtime' -- just Restart session, it "
      "keeps the same GPU). Do NOT re-run this cell after restarting. "
      "Then run cell-01 again (quick GPU check) and continue with the "
      "next cell below, which only imports and verifies -- it does not "
      "reinstall anything.")

In [ ]:
# ---- Part D, step 2b: verify imports (run AFTER restarting the session) ----
# This cell only imports and checks versions -- it must not reinstall
# anything, so a partially-broken import state can't be masked by a second
# install in the same process.
import transformers, peft, trl, bitsandbytes, accelerate, triton
import torch

for m in (transformers, peft, trl, bitsandbytes, accelerate, triton):
    print(f"{m.__name__:15s} {m.__version__}")
print(f"{'torch':15s} {torch.__version__}")

assert torch.cuda.is_available(), (
    "torch reports no CUDA after restart -- re-check Runtime > Change "
    "runtime type still shows GPU: L4, then restart again.")

try:
    from bitsandbytes.cextension import CUDASetup
    setup = CUDASetup.get_instance()
    cuda_available = setup.cuda_available
    print(f"\nbitsandbytes CUDA available: {cuda_available}")
    if not cuda_available:
        print("WARNING: bitsandbytes did not find CUDA even after restart.")
except Exception:
    print(f"\nbitsandbytes status: Ready (Torch CUDA: {torch.cuda.is_available()})")

print("\nAll imports clean -- continue to the Data section below.")

## Data

Four files produced by Parts A/B/C on the CPU machine are required:

| File | Produced by |
|---|---|
| `dataset.jsonl` | `data/build_dataset.py` (3,760 sandbox-verified MBPP repair pairs) |
| `dpo_pairs.jsonl` | `data/build_dataset.py` (chosen = reference fix, rejected = a different variant already verified to fail) |
| `humaneval_broken_with_context.jsonl` | `data/build_eval_set.py` + `data/add_context_to_eval.py` (held-out eval set, retrieval context precomputed by the real RAG pipeline) |
| `executor.py` | Part A sandbox -- reused as-is for evaluation, not reimplemented |

Preferred path: copy them to `MyDrive/code-repair/` before running. The next
cell mounts Drive and looks there; the cell after it is a manual-upload
fallback.

In [8]:
# ---- Part D, step 3: locate data (Drive first) ----
import os

REQUIRED = ["dataset.jsonl", "dpo_pairs.jsonl",
            "humaneval_broken_with_context.jsonl", "executor.py"]
DATA_DIR = None
BACKUP_DIR = "/content/adapter_backups"  # overridden if Drive is available

try:
    from google.colab import drive
    drive.mount("/content/drive")
    candidate = "/content/drive/MyDrive/code-repair"
    if all(os.path.exists(os.path.join(candidate, f)) for f in REQUIRED):
        DATA_DIR = candidate
    BACKUP_DIR = "/content/drive/MyDrive/code-repair/adapters"
except Exception as exc:
    print(f"Drive not available ({exc}); manual upload fallback required.")

if DATA_DIR is None and all(os.path.exists(f"/content/{f}") for f in REQUIRED):
    DATA_DIR = "/content"   # files were uploaded manually

os.makedirs(BACKUP_DIR, exist_ok=True)
if DATA_DIR:
    print(f"data directory : {DATA_DIR}")
    print(f"adapter backups: {BACKUP_DIR}")
else:
    print("MISSING FILES. Either copy these to MyDrive/code-repair/ and "
          "re-run this cell, or run the manual-upload cell below:\n  "
          + "\n  ".join(REQUIRED))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data directory : /content/drive/MyDrive/code-repair
adapter backups: /content/drive/MyDrive/code-repair/adapters


In [9]:
# ---- Manual-upload fallback: run ONLY if the previous cell reported
# ---- missing files. Select all four required files in the picker.
if DATA_DIR is None:
    from google.colab import files
    uploaded = files.upload()
    print(f"uploaded: {list(uploaded)}")
    missing = [f for f in REQUIRED if not os.path.exists(f"/content/{f}")]
    if missing:
        raise SystemExit(f"still missing: {missing}")
    DATA_DIR = "/content"
    print("all required files present in /content")
else:
    print("skipped -- data already found")

skipped -- data already found


In [10]:
# ---- Part D, step 4: load the dataset (real counts printed) ----
import json, random

def load_jsonl(name):
    with open(os.path.join(DATA_DIR, name), encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_pairs = load_jsonl("dataset.jsonl")
dpo_pairs = load_jsonl("dpo_pairs.jsonl")
eval_items = load_jsonl("humaneval_broken_with_context.jsonl")

rng = random.Random(13)
rng.shuffle(train_pairs)
n_val = max(64, int(0.02 * len(train_pairs)))
val_pairs, sft_pairs = train_pairs[:n_val], train_pairs[n_val:]

print(f"training pairs : {len(sft_pairs)} (val: {len(val_pairs)})")
print(f"dpo pairs      : {len(dpo_pairs)}")
print(f"eval items     : {len(eval_items)} (held-out HumanEval, never trained on)")
assert all(i["source"] == "humaneval" for i in eval_items)
assert all(p["source"] == "mbpp" for p in train_pairs)

training pairs : 3685 (val: 75)
dpo pairs      : 3752
eval items     : 467 (held-out HumanEval, never trained on)


In [11]:
# ---- Part D, step 5: prompt format (shared by training and eval) ----
SYSTEM_PROMPT = (
    "You are a precise Python code repair assistant. Given a problem "
    "description, a broken solution and the error it produces, reply with "
    "the corrected, complete Python code in a single fenced code block and "
    "nothing else.")

def user_prompt(item, use_context=False):
    parts = []
    ctx = item.get("retrieved_context") if use_context else None
    if ctx:
        parts.append("Similar past repairs for reference:\n\n" + ctx)
    parts.append("Problem:\n" + item["problem"].strip())
    parts.append("Broken code:\n```python\n" + item["broken_code"] + "\n```")
    parts.append("Error produced by the tests:\n```\n" + item["error"] + "\n```")
    return "\n\n".join(parts)

def assistant_answer(fixed_code):
    return "```python\n" + fixed_code + "\n```"

def to_chat_text(tokenizer, item, use_context=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(item, use_context)},
        {"role": "assistant", "content": assistant_answer(item["fixed_code"])},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

print("prompt example (truncated to 900 chars):\n")
print(user_prompt(train_pairs[0])[:900])

prompt example (truncated to 900 chars):

Problem:
Write a function that matches a string that has an a followed by two to three 'b'.

Broken code:
```python
import re

def text_match_two_three(text):
    patterns = 'ab{2,3}'
    if re.search(text, text):
        return 'Found a match!'
    else:
        return 'Not matched!'
```

Error produced by the tests:
```
Traceback (most recent call last):
  File "test_case_0.py", line 1, in <module>
AssertionError
```


## Base model choice: Qwen2.5-Coder-**1.5B**-Instruct

Qwen2.5-Coder ships in 0.5B / 1.5B / 3B / 7B / 14B / 32B. **7B is the
largest size for which LoRA, QLoRA, and DoRA can all run on one L4 in
bf16** (~15.2 GB of weights, fitting in roughly 18-21 GB with gradient
checkpointing) -- that was the original choice and it does work. A later
pass used 3B as a time trade-off against a hard deadline.

This run uses **1.5B**, for two reasons together, not hardware limits:

1. **Speed.** 1.5B in bf16 is ~3 GB of weights -- the full LoRA / QLoRA /
   DoRA / DPO / five-arm-benchmark pipeline completes in well under an
   hour of GPU time, comfortably inside any time budget.
2. **A clean apples-to-apples comparison.** The web UI (Part F) already
   runs the *base*, un-fine-tuned `qwen2.5-coder:1.5b` through Ollama for
   its live demo. Fine-tuning the same 1.5B model means the benchmark's
   "base" row and the UI's current demo model are the exact same weights
   at the exact same size -- so the pass@1/pass@3 lift measured in Part E
   is not just "a bigger model did better," it is "the same-size model got
   better from this pipeline." Swapping the UI to the fine-tuned adapter
   afterward is a like-for-like upgrade, not a size change too.

**If asked in an interview:** "I sized this at different points for
different reasons. 7B is the largest model all three PEFT methods
comfortably fit on one L4 in bf16. For this run I used 1.5B specifically
because it's the same size as the base model already wired into my UI --
so the benchmark and the live demo are directly comparable, and it trains
fast enough to run the whole pipeline -- LoRA, QLoRA, DoRA, DPO, and the
full benchmark -- well inside an hour."

In [ ]:
# ---- Part D, step 6: shared training machinery ----
import gc, shutil, time
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig)
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
RESULTS = {}   # real measured numbers accumulate here

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

def free_gpu(*objs):
    for o in objs:
        try:
            del o
        except NameError:
            pass
    gc.collect()
    torch.cuda.empty_cache()
    print(f"gpu allocated after cleanup: "
          f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB")

def load_base(four_bit=False):
    kwargs = dict(torch_dtype=torch.bfloat16, device_map="auto",
                  attn_implementation="sdpa")
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
    if four_bit:
        model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False
    return model

def make_sft_dataset(pairs):
    return Dataset.from_dict(
        {"text": [to_chat_text(tokenizer, p) for p in pairs]})

sft_train_ds = make_sft_dataset(sft_pairs)
sft_val_ds = make_sft_dataset(val_pairs)
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False),
    tokenizer=tokenizer)

def backup_to_drive(local_dir, run_name):
    dest = os.path.join(BACKUP_DIR, run_name)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(local_dir, dest)
    print(f"adapter backed up to {dest}")

def run_sft(run_name, four_bit=False, use_dora=False):
    print(f"===== SFT run: {run_name} "
          f"(four_bit={four_bit}, dora={use_dora}) =====")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.monotonic()
    model = load_base(four_bit=four_bit)
    peft_config = LoraConfig(
        task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05,
        use_dora=use_dora,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"])
    out_dir = f"/content/adapters/{run_name}"
    args = SFTConfig(
        output_dir=out_dir, num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=LEARNING_RATE, lr_scheduler_type="cosine",
        warmup_ratio=0.03, bf16=True, gradient_checkpointing=True,
        logging_steps=20, eval_strategy="epoch", save_strategy="no",
        max_seq_length=MAX_SEQ_LEN, dataset_text_field="text",
        packing=False, report_to="none", seed=13)
    trainer = SFTTrainer(model=model, args=args, train_dataset=sft_train_ds,
                         eval_dataset=sft_val_ds, data_collator=collator,
                         peft_config=peft_config, processing_class=tokenizer)
    train_out = trainer.train()
    eval_out = trainer.evaluate()
    wall_min = (time.monotonic() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3

    trainer.save_model(out_dir)          # adapter only (peft)
    backup_to_drive(out_dir, run_name)   # to Drive IMMEDIATELY, not at the end

    RESULTS[run_name] = {
        "train_loss": round(train_out.metrics["train_loss"], 4),
        "eval_loss": round(eval_out["eval_loss"], 4),
        "wall_minutes": round(wall_min, 1),
        "peak_vram_gb": round(peak_gb, 2),
        "adapter": out_dir,
    }
    print(f"{run_name}: train_loss={RESULTS[run_name]['train_loss']} "
          f"eval_loss={RESULTS[run_name]['eval_loss']} "
          f"time={wall_min:.1f} min peak_vram={peak_gb:.2f} GB")
    free_gpu(trainer, model)
    return RESULTS[run_name]

### LoRA (bf16 base)

In [ ]:
lora_metrics = run_sft("lora")

===== SFT run: lora (four_bit=False, dora=False) =====


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Map:   0%|          | 0/3685 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss


### QLoRA (4-bit NF4 base)

Same adapter shape and hyperparameters; only the base quantization differs,
so the comparison isolates that variable.

In [ ]:
qlora_metrics = run_sft("qlora", four_bit=True)

### DoRA (weight-decomposed LoRA, bf16 base)

In [ ]:
dora_metrics = run_sft("dora", use_dora=True)

## DPO on the best SFT adapter

`dpo_pairs.jsonl` was built in Part B: **chosen** is the real reference fix;
**rejected** is a *different* bug variant of the same problem -- a genuinely
plausible-looking repair that was already executed in the Part A sandbox at
dataset-build time and verified to fail the problem's real tests. The next
cell re-verifies a sample of 20 rejected answers in the sandbox here, so the
claim is checked on this machine too, not taken on faith.

In [ ]:
# ---- re-verify a sample of DPO 'rejected' answers really fail ----
import sys
sys.path.insert(0, DATA_DIR)
from executor import run_tests   # the Part A sandbox, reused verbatim

tests_by_task = {}
for p in train_pairs:
    tests_by_task.setdefault(p["task_id"], (p["tests"], p["test_setup"]))

sample = random.Random(5).sample(dpo_pairs, 20)
still_failing = 0
for pair in sample:
    tests, setup = tests_by_task[pair["task_id"]]
    r = run_tests(pair["rejected"], tests, setup_code=setup, timeout_s=8.0)
    still_failing += (not r.ok)
print(f"re-verified: {still_failing}/20 sampled rejected answers fail "
      f"their problem's real tests")
assert still_failing == 20, "a rejected answer passed; investigate before DPO"


In [ ]:
# ---- DPO training ----
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

best_sft = min(("lora", "qlora", "dora"), key=lambda k: RESULTS[k]["eval_loss"])
print(f"best SFT adapter by measured eval_loss: {best_sft} "
      f"(eval_loss={RESULTS[best_sft]['eval_loss']})")

DPO_SUBSET = 2000   # pairs; reduce for a faster run
dpo_subset = random.Random(17).sample(dpo_pairs,
                                      min(DPO_SUBSET, len(dpo_pairs)))

def to_dpo_row(pair):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt(pair)}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True)
    return {"prompt": prompt,
            "chosen": assistant_answer(pair["chosen"]),
            "rejected": assistant_answer(pair["rejected"])}

dpo_ds = Dataset.from_list([to_dpo_row(p) for p in dpo_subset])
print(f"dpo training rows: {len(dpo_ds)}")

torch.cuda.reset_peak_memory_stats()
t0 = time.monotonic()
base = load_base(four_bit=(best_sft == "qlora"))
model = PeftModel.from_pretrained(base, RESULTS[best_sft]["adapter"],
                                  is_trainable=True)

dpo_args = DPOConfig(
    output_dir="/content/adapters/dpo", num_train_epochs=1, beta=0.1,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=5e-6, lr_scheduler_type="cosine", warmup_ratio=0.03,
    bf16=True, gradient_checkpointing=True, logging_steps=20,
    save_strategy="no", max_length=1536, max_prompt_length=1152,
    report_to="none", seed=13)
dpo_trainer = DPOTrainer(model=model, ref_model=None, args=dpo_args,
                         train_dataset=dpo_ds, processing_class=tokenizer)
dpo_out = dpo_trainer.train()
wall_min = (time.monotonic() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1024**3

dpo_trainer.save_model("/content/adapters/dpo")
backup_to_drive("/content/adapters/dpo", "dpo")

RESULTS["dpo"] = {
    "train_loss": round(dpo_out.metrics["train_loss"], 4),
    "eval_loss": None,
    "wall_minutes": round(wall_min, 1),
    "peak_vram_gb": round(peak_gb, 2),
    "adapter": "/content/adapters/dpo",
    "base_sft": best_sft,
}
print(f"dpo: train_loss={RESULTS['dpo']['train_loss']} "
      f"time={wall_min:.1f} min peak_vram={peak_gb:.2f} GB "
      f"(on top of {best_sft})")
free_gpu(dpo_trainer, model, base)

# Part E -- Benchmark on held-out HumanEval

Protocol:
- Evaluation set: the HumanEval-derived broken-code items (never used in
  training in any form). `N_EVAL` items are drawn with a fixed seed.
- **pass@1**: one greedy generation per item; the fix counts only if it
  passes the problem's real `check()` tests **executed in the Part A
  sandbox** (the same `executor.py`, imported, not reimplemented).
- **pass@3**: three samples at temperature 0.8; counts if any passes.
- RAG ablation: the same items are run with and without the retrieval
  context that the real Part C pipeline precomputed for each item, so the
  effect of RAG+GraphRAG is isolated from the model variable.

Deterministic, execution-based -- no LLM judging anywhere.

In [ ]:
# ---- Part E, step 1: sandbox smoke check on this machine ----
good = run_tests("def add(a, b):\n    return a + b\n",
                 "def check(candidate):\n    assert candidate(1, 2) == 3\n",
                 entry_point="add")
bad = run_tests("def add(a, b):\n    return a - b\n",
                "def check(candidate):\n    assert candidate(1, 2) == 3\n",
                entry_point="add")
assert good.ok and not bad.ok
print("sandbox operational on this runtime (POSIX rlimit branch)")

In [ ]:
# ---- Part E, step 2: generation + execution harness ----
import re
from concurrent.futures import ThreadPoolExecutor

N_EVAL = 150          # items per arm; raise to len(eval_items) for the full set
PASS_AT_K_SAMPLES = 3
MAX_NEW_TOKENS = 640
GEN_BATCH = 8

eval_subset = random.Random(23).sample(eval_items,
                                       min(N_EVAL, len(eval_items)))
print(f"evaluation subset: {len(eval_subset)} of {len(eval_items)} items")

_CODE_RE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.DOTALL)

def extract_code(text):
    blocks = _CODE_RE.findall(text)
    return (max(blocks, key=len) if blocks else text).strip()

def build_eval_prompts(items, use_context):
    prompts = []
    for item in items:
        messages = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",
                     "content": user_prompt(item, use_context=use_context)}]
        prompts.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True))
    return prompts

def generate(model, prompts, do_sample, temperature=0.8):
    outs = []
    tokenizer.padding_side = "left"
    for start in range(0, len(prompts), GEN_BATCH):
        batch = prompts[start:start + GEN_BATCH]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            gen = model.generate(
                **enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=do_sample,
                temperature=temperature if do_sample else None,
                top_p=0.95 if do_sample else None,
                pad_token_id=tokenizer.pad_token_id)
        for i in range(len(batch)):
            new_tokens = gen[i][enc["input_ids"].shape[1]:]
            outs.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return outs

def execute_fix(args):
    item, fix_code = args
    r = run_tests(fix_code, item["test_code"],
                  entry_point=item["entry_point"], timeout_s=8.0)
    return r.ok

def evaluate_arm(tag, adapter=None, four_bit=False, use_context=False):
    print(f"===== eval arm: {tag} =====")
    t0 = time.monotonic()
    model = load_base(four_bit=four_bit)
    if adapter:
        model = PeftModel.from_pretrained(model, adapter)
    model.eval()
    model.config.use_cache = True

    prompts = build_eval_prompts(eval_subset, use_context)

    greedy = generate(model, prompts, do_sample=False)
    samples = [generate(model, prompts, do_sample=True)
               for _ in range(PASS_AT_K_SAMPLES)]
    free_gpu(model)

    with ThreadPoolExecutor(max_workers=8) as pool:
        greedy_ok = list(pool.map(execute_fix,
            [(it, extract_code(g)) for it, g in zip(eval_subset, greedy)]))
        sample_ok = []
        for s in samples:
            sample_ok.append(list(pool.map(execute_fix,
                [(it, extract_code(g)) for it, g in zip(eval_subset, s)])))

    n = len(eval_subset)
    pass1 = sum(greedy_ok) / n
    pass3 = sum(any(col) for col in zip(*sample_ok)) / n
    wall_min = (time.monotonic() - t0) / 60
    EVAL_RESULTS[tag] = {"pass@1": round(pass1, 4), "pass@3": round(pass3, 4),
                         "n": n, "minutes": round(wall_min, 1),
                         "rag_context": use_context}
    print(f"{tag}: pass@1={pass1:.3f} pass@3={pass3:.3f} "
          f"({n} items, {wall_min:.1f} min)")
    return EVAL_RESULTS[tag]

EVAL_RESULTS = {}

In [ ]:
# ---- Part E, step 3: run every arm (each frees GPU before the next) ----
evaluate_arm("base")
evaluate_arm("lora", adapter=RESULTS["lora"]["adapter"])
evaluate_arm("qlora", adapter=RESULTS["qlora"]["adapter"], four_bit=True)
evaluate_arm("dora", adapter=RESULTS["dora"]["adapter"])
evaluate_arm("dpo", adapter=RESULTS["dpo"]["adapter"],
             four_bit=(RESULTS["dpo"]["base_sft"] == "qlora"))

In [ ]:
# ---- Part E, step 4: RAG ablation (same items, context on) ----
best_tag = max(("lora", "qlora", "dora", "dpo"),
               key=lambda k: EVAL_RESULTS[k]["pass@1"])
print(f"best arm by measured pass@1: {best_tag}")

evaluate_arm("base+RAG", use_context=True)
evaluate_arm(f"{best_tag}+RAG", adapter=RESULTS[best_tag]["adapter"],
             four_bit=(best_tag == "qlora" or
                       (best_tag == "dpo" and
                        RESULTS["dpo"]["base_sft"] == "qlora")),
             use_context=True)

In [ ]:
# ---- Part E, step 5: combined results table (all values measured) ----
import pandas as pd

rows = []
for tag, m in EVAL_RESULTS.items():
    train = RESULTS.get(tag.replace("+RAG", ""), {})
    rows.append({
        "arm": tag,
        "pass@1": m["pass@1"],
        "pass@3": m["pass@3"],
        "RAG context": "yes" if m["rag_context"] else "no",
        "eval items": m["n"],
        "train eval_loss": train.get("eval_loss"),
        "train minutes": train.get("wall_minutes"),
        "train peak VRAM (GB)": train.get("peak_vram_gb"),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))

table.to_csv("/content/benchmark_results.csv", index=False)
if "drive" in BACKUP_DIR:
    shutil.copy("/content/benchmark_results.csv",
                os.path.join(os.path.dirname(BACKUP_DIR),
                             "benchmark_results.csv"))
    print("results table copied to Drive")

## Export: merge best adapter, quantize to GGUF (q4_k_m), Modelfile

Produces `code-repair-qwen-q4_k_m.gguf` plus a `Modelfile` ready for
`ollama create code-repair-qwen -f Modelfile` on the local machine. The UI
then switches to the fine-tuned model by changing `OLLAMA_MODEL` in
`ui/config.py` -- one line.

In [ ]:
# ---- merge the best adapter into the base model (bf16, on CPU RAM) ----
export_tag = best_tag
adapter_dir = RESULTS[export_tag.replace("+RAG", "")]["adapter"]
print(f"exporting arm: {export_tag} (adapter: {adapter_dir})")

merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu")
merged = PeftModel.from_pretrained(merge_base, adapter_dir)
merged = merged.merge_and_unload()
merged.save_pretrained("/content/merged", safe_serialization=True)
tokenizer.save_pretrained("/content/merged")
del merged, merge_base
gc.collect()
print("merged model saved to /content/merged")

In [ ]:
# ---- build llama.cpp (installs cmake if missing) and quantize ----
import shutil as _sh
if _sh.which("cmake") is None:
    !apt-get -qq install -y cmake
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
%pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null

!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged \
    --outfile /content/code-repair-qwen-f16.gguf --outtype f16
!/content/llama.cpp/build/bin/llama-quantize \
    /content/code-repair-qwen-f16.gguf \
    /content/code-repair-qwen-q4_k_m.gguf q4_k_m

import os
for f in ("code-repair-qwen-f16.gguf", "code-repair-qwen-q4_k_m.gguf"):
    path = f"/content/{f}"
    if os.path.exists(path):
        print(f"{f}: {os.path.getsize(path) / 1024**3:.2f} GB")

In [ ]:
# ---- Modelfile + copy artifacts to Drive ----
MODELFILE = '''FROM ./code-repair-qwen-q4_k_m.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

PARAMETER stop <|im_end|>
PARAMETER temperature 0.2
SYSTEM You are a precise Python code repair assistant. Reply with the corrected, complete Python code in a single fenced code block and nothing else.
'''
with open("/content/Modelfile", "w") as f:
    f.write(MODELFILE)
print("wrote /content/Modelfile")

if "drive" in BACKUP_DIR:
    dest_dir = os.path.dirname(BACKUP_DIR)
    for f in ("code-repair-qwen-q4_k_m.gguf", "Modelfile"):
        _sh.copy(f"/content/{f}", os.path.join(dest_dir, f))
        print(f"copied {f} to {dest_dir}")
else:
    print("Drive not mounted -- download the GGUF and Modelfile manually "
          "from the Files panel before the runtime recycles.")

## Done

On the local machine (see RUNBOOK.md):

```
ollama create code-repair-qwen -f Modelfile
```

then set `OLLAMA_MODEL = "code-repair-qwen"` in `ui/config.py` and restart
the UI server. Every number this notebook printed was measured during your
run; if a cell was skipped or failed, its numbers do not exist -- there are
no defaults to fall back on.